In [2]:
import torch
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment


In [18]:
feature_path = "/mnt/abka03/concept_extraction_result/publish/gemma3n/CGDL/SNMF/imagenet1000/train/concept/combined_concept_sae2_raw.pth"
features = torch.load(feature_path)["concepts"]
print(features.shape)



torch.Size([200, 2048])


In [20]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import KFold

# === Metrics ===

def compute_overlap(Z):
    Z_bin = (Z != 0).float()
    coactivation = (Z_bin @ Z_bin.T) / Z.shape[1]
    return coactivation

def compute_single_overlap(Z):
    Z_bin = (Z != 0).float()
    coactivation = (Z_bin @ Z_bin.T) / Z.shape[1]  # (k, k)
    k = coactivation.shape[0]
    mask = torch.ones_like(coactivation) - torch.eye(k)
    overlap_score = (coactivation * mask).sum() / (k * (k - 1))
    return overlap_score.item()

def compute_sparsity(Z):
    total_elements = Z.numel()
    nonzeros = (Z != 0).sum().item()
    sparsity = 1.0 - (nonzeros / total_elements)
    return sparsity

def compute_stability_single(Z1, Z2):
    assert Z1.shape == Z2.shape, "Z1 and Z2 must have the same shape"
    Z1_norm = F.normalize(Z1, p=2, dim=1)
    Z2_norm = F.normalize(Z2, p=2, dim=1)
    cos_sim = (Z1_norm * Z2_norm).sum(dim=1)
    return cos_sim.mean().item()

def example_extractor(A_fold):
    """
    A_fold: (n_samples_fold, n_features), torch tensor
    Returns: (U, V) with V: (n_components, n_features)
    """
    A_fold = A_fold.float()
    m, n = A_fold.shape
    q = min(m, n, 10)  # choose q <= min(m, n)
    U, S, V = torch.pca_lowrank(A_fold, q=q)
    return U, V.T  

def compute_stability_kfold(A, extractor_fn, method_name, n_folds=2):
    """
    A: (n_samples, n_features) as a NumPy array or Torch tensor
    extractor_fn: returns (U, V) where V is (n_components, n_features)
    """
    if isinstance(A, torch.Tensor):
        A = A.cpu().numpy()
        
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    concept_banks = []

    for _, idx in kf.split(A):
        A_fold = A[idx]
        U, V = extractor_fn(torch.tensor(A_fold))
        if isinstance(V, torch.Tensor):
            V = V.cpu().numpy()

        if np.isnan(V).any() or np.isinf(V).any():
            raise ValueError("Concept matrix contains NaN or Inf values.")

        concept_banks.append(V)

    similarities = []
    for i in range(n_folds):
        for j in range(i + 1, n_folds):
            V1, V2 = concept_banks[i], concept_banks[j]

            if V1.shape != V2.shape:
                raise ValueError(f"Shape mismatch: {V1.shape} vs {V2.shape}")

            sim_matrix = cosine_similarity(V1, V2)

            if np.isnan(sim_matrix).any() or np.isinf(sim_matrix).any():
                raise ValueError(f"Similarity matrix contains invalid values.")

            row_ind, col_ind = linear_sum_assignment(-sim_matrix)
            matched_sims = sim_matrix[row_ind, col_ind]
            similarities.append(np.mean(matched_sims))

    if not similarities:
        raise RuntimeError("No similarities computed.")

    mean_similarity = np.mean(similarities)
    stability = 1 - mean_similarity
    print(f"Stability ({method_name}): {stability:.4f} ↓")
    return stability

# === Dummy concept extractor using SVD ===

def example_extractor(A_fold):
    """
    A_fold: (n_samples_fold, n_features), torch tensor
    Returns: (U, V) with V: (n_components, n_features)
    """
    A_fold = A_fold.float()
    U, S, V = torch.pca_lowrank(A_fold, q=10)
    return U, V.T  # V.T to match (n_components, n_features)

# === Main Script ===

if __name__ == "__main__":
    torch.manual_seed(0)

    # Simulate 10 samples, 2048 features (Z: 2048 x 10)
   
    Z = features.T  # (2048, 10)

    # Sparsify Z
    Z[Z.abs() < 0.5] = 0

    # Compute metrics
    overlap_score = compute_single_overlap(Z)
    sparsity_score = compute_sparsity(Z)

    # Simulate a noisy version for pairwise stability
    Z1 = Z.clone()
    Z2 = Z1 + 0.1 * torch.randn_like(Z1)
    Z1[Z1.abs() < 0.5] = 0
    Z2[Z2.abs() < 0.5] = 0
    stability_single = compute_stability_single(Z1, Z2)

    # Print metrics
    print("=== Overlap (single scalar) ===")
    print(f"{overlap_score:.4f}")

    print("\n=== Sparsity (fraction of zeros) ===")
    print(f"{sparsity_score:.4f}")

    print("\n=== Stability (mean cosine similarity) ===")
    print(f"{stability_single:.4f}")

    # K-Fold stability (on original data, shape must be (samples, features))
    A = Z.T  # shape: (10, 2048)
    compute_stability_kfold(A, extractor_fn=example_extractor, method_name="SVD", n_folds=3)


=== Overlap (single scalar) ===
0.0753

=== Sparsity (fraction of zeros) ===
0.7285

=== Stability (mean cosine similarity) ===
0.9100
Stability (SVD): 0.9699 ↓
